In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import os
from PIL import Image

from torchvision import datasets,transforms
from torch.utils.data import Dataset,DataLoader
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split

#Download the dataset from Kaggle and place the 'PetImages' folder in your desired path.
#Then update the 'base' variable in the notebook.

base = "PetImages"
if not os.path.exists(base):
    print("Dataset folder not found. Please update the 'base' variable.")
"""
# Optional: remove corrupted images if needed:
for folder in ["Cat","Dog"]:
    path = os.path.join(base,folder)
    for file in os.listdir(path):
        fpath = os.path.join(path,file)
        try:
            img=Image.open(fpath)
            img.verify()
        except:
            print("removing:",fpath)
            os.remove(fpath)
"""

transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2,contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])

dataset=ImageFolder(root=base,transform=transform)
train_size=int(0.8*len(dataset))
test_size=len(dataset) - train_size
train_dataset,test_dataset=random_split(dataset,[train_size,test_size])

batch_size = 64
train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False)




# Select device (GPU if available, otherwise CPU)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device = ",device)

# Build CNN Model

In [ ]:
# Define a CNN architecture for binary classification (Cat vs Dog)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(128,256,kernel_size=3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )
        self.conv5 = nn.Sequential(
            nn.Conv2d(256,256,kernel_size=3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )


        self.pool = nn.MaxPool2d(2,2)

        self.fc1 = nn.Linear(256*4*4,512)
        self.dropout = nn.Dropout(0.5) # prevent overfitting
        self.fc2 = nn.Linear(512,2)

    def forward(self,x):
        x = self.pool(self.conv1(x))
        x = self.pool(self.conv2(x))
        x = self.pool(self.conv3(x))
        x = self.pool(self.conv4(x))
        x = self.pool(self.conv5(x))
        x = x.view(x.size(0),-1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)     # Dropout
        x = self.fc2(x)
        return x

model = CNN().to(device) 

# Define Loss Function and Optimizer


In [ ]:
criterion = nn.CrossEntropyLoss()    # CrossEntropyLoss for binary classification (Cat vs Dog)
optimizer = optim.AdamW(model.parameters(),lr=0.001)  # AdamW optimizer with learning rate 0.001

# Train the Model


In [ ]:
loss_history = []
epochs = 20

# Training loop for CNN model
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")


# Evaluate the Model


In [ ]:
# Compute accuracy on the test dataset
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images , labels in test_loader:
        images , labels = images.to(device),labels.to(device)
        outputs = model(images)
        _,predicted = torch.max(outputs.data,1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
print("Test Accuracy = ",correct/total)

# Plot Training Loss


In [ ]:
# Visualize the training loss over epochs
import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))
plt.plot(loss_history,color='blue')
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

# Make Predictions on Sample Images


In [ ]:
# Load sample images, preprocess them, and run inference
from PIL import Image
# Place your test images (1.jpg, 2.jpg, ...) in the same folder as the notebook
image1 = Image.open("1.jpg")
image2 = Image.open("2.jpg")
image3 = Image.open("3.jpg")
image4 = Image.open("4.jpg")

images = [transform(image1), transform(image2), transform(image3), transform(image4)]
batch = torch.stack(images).to(device)
model.eval()
with torch.no_grad():
    outputs = model(batch)
    _,preds = torch.max(outputs,1)

classes = dataset.classes
for i,p in enumerate(preds):
    print(f"Image {i+1}: {classes[p.item()]}")
